# NB02: Literature-Informed Marker Gene Survey

Search BERDL annotations for established plant-microbe interaction markers across three functional categories:
- **Beneficial/PGP**: nitrogen fixation, phosphate solubilization, IAA, ACC deaminase, siderophores, biocontrol
- **Pathogenic**: type III/IV/VI secretion systems, effectors, toxins, cell wall degrading enzymes
- **Colonization**: flagella, chemotaxis, quorum sensing, exopolysaccharides

Species are classified into PGP-only, pathogen-only, dual-nature, and neutral cohorts.

**Requires**: Spark (on BERDL JupyterHub)

**Outputs**: `data/marker_gene_clusters.csv`, `data/species_marker_matrix.csv`, `data/species_cohort_markers.csv`

In [ ]:
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))
elif os.path.exists(os.path.join(_here, 'projects', 'plant_microbiome_ecotypes')):
    REPO = _here
else:
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))

PROJECT = os.path.join(REPO, 'projects', 'plant_microbiome_ecotypes')
DATA = os.path.join(PROJECT, 'data')
FIGURES = os.path.join(PROJECT, 'figures')

os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')

## 1. Define marker gene sets

Three search strategies:
- **bakta_annotations.gene**: exact gene name matches
- **bakta_annotations.product**: keyword search in product descriptions
- **eggnog_mapper_annotations.KEGG_ko**: KEGG orthology IDs
- **bakta_pfam_domains.pfam_id**: Pfam domain matches

In [ ]:
# === BENEFICIAL / PGP MARKERS ===
PGP_GENES = {
    # Nitrogen fixation
    'nifH': 'nitrogen_fixation', 'nifD': 'nitrogen_fixation', 'nifK': 'nitrogen_fixation',
    # ACC deaminase
    'acdS': 'acc_deaminase',
    # Phosphate solubilization (pyrroloquinoline quinone)
    'pqqA': 'phosphate_solubilization', 'pqqB': 'phosphate_solubilization',
    'pqqC': 'phosphate_solubilization', 'pqqD': 'phosphate_solubilization',
    'pqqE': 'phosphate_solubilization',
    # IAA biosynthesis
    'ipdC': 'iaa_biosynthesis',
    # Hydrogen cyanide (biocontrol)
    'hcnA': 'hydrogen_cyanide', 'hcnB': 'hydrogen_cyanide', 'hcnC': 'hydrogen_cyanide',
    # Siderophore biosynthesis
    'entA': 'siderophore', 'entB': 'siderophore', 'entC': 'siderophore',
    'entD': 'siderophore', 'entE': 'siderophore', 'entF': 'siderophore',
    'pvdA': 'siderophore', 'pvdS': 'siderophore',
    # Acetoin / 2,3-butanediol (ISR)
    'budA': 'acetoin_butanediol', 'budB': 'acetoin_butanediol', 'budC': 'acetoin_butanediol',
    'alsD': 'acetoin_butanediol', 'alsS': 'acetoin_butanediol',
    # DAPG (biocontrol)
    'phlA': 'dapg_biocontrol', 'phlB': 'dapg_biocontrol',
    'phlC': 'dapg_biocontrol', 'phlD': 'dapg_biocontrol',
    # Phenazine (biocontrol)
    'phzA': 'phenazine', 'phzB': 'phenazine', 'phzC': 'phenazine',
    'phzD': 'phenazine', 'phzE': 'phenazine', 'phzF': 'phenazine', 'phzG': 'phenazine',
    # Biofilm (plant colonization support)
    'bcsA': 'biofilm', 'pelA': 'biofilm',
}

# === PATHOGENIC MARKERS ===
PATHOGEN_GENES = {
    # Type III secretion system (T3SS)
    'hrcC': 't3ss', 'hrcJ': 't3ss', 'hrcN': 't3ss', 'hrcV': 't3ss',
    'hrpA': 't3ss', 'hrpB': 't3ss', 'hrpL': 't3ss',
    'sctC': 't3ss', 'sctJ': 't3ss', 'sctN': 't3ss', 'sctV': 't3ss',
    # Type IV secretion system (T4SS)
    'virB1': 't4ss', 'virB2': 't4ss', 'virB4': 't4ss', 'virB6': 't4ss',
    'virB8': 't4ss', 'virB9': 't4ss', 'virB10': 't4ss', 'virB11': 't4ss',
    'virD2': 't4ss', 'virD4': 't4ss',
    # Type VI secretion system (T6SS)
    'tssB': 't6ss', 'tssC': 't6ss', 'tssE': 't6ss', 'tssF': 't6ss',
    'tssG': 't6ss', 'tssH': 't6ss', 'tssK': 't6ss', 'tssM': 't6ss',
    # Type II secretion system (T2SS)
    'gspD': 't2ss', 'gspE': 't2ss', 'gspF': 't2ss',
    # Coronatine toxin
    'cmaA': 'coronatine_toxin', 'cmaB': 'coronatine_toxin',
    'cfa6': 'coronatine_toxin', 'cfa7': 'coronatine_toxin',
    # Cell wall degrading enzymes
    'pelA': 'cwde_pectinase', 'pelB': 'cwde_pectinase', 'pelC': 'cwde_pectinase',
    'pelD': 'cwde_pectinase', 'pelE': 'cwde_pectinase',
    'pehA': 'cwde_pectinase',
    'celA': 'cwde_cellulase', 'celB': 'cwde_cellulase',
}

# === COLONIZATION MARKERS ===
COLONIZATION_GENES = {
    'fliC': 'flagella', 'flgE': 'flagella', 'flhA': 'flagella',
    'cheA': 'chemotaxis', 'cheW': 'chemotaxis', 'cheR': 'chemotaxis', 'cheY': 'chemotaxis',
    'luxI': 'quorum_sensing', 'luxR': 'quorum_sensing',
}

# Combine all gene names for query
ALL_MARKER_GENES = {**PGP_GENES, **PATHOGEN_GENES, **COLONIZATION_GENES}
all_gene_names = sorted(set(ALL_MARKER_GENES.keys()))
print(f'Total marker genes to search: {len(all_gene_names)}')
print(f'  PGP: {len(PGP_GENES)} genes')
print(f'  Pathogenic: {len(PATHOGEN_GENES)} genes')
print(f'  Colonization: {len(COLONIZATION_GENES)} genes')

## 2. Query bakta_annotations for gene name matches

In [ ]:
# Search bakta_annotations by gene name
bakta_markers_path = os.path.join(DATA, 'bakta_marker_hits.csv')
if os.path.exists(bakta_markers_path):
    bakta_hits = pd.read_csv(bakta_markers_path)
    print(f'Bakta marker hits (loaded from cache): {len(bakta_hits):,}')
else:
    spark = get_spark_session()
    # Build IN clause for gene names
    gene_in = "', '".join(all_gene_names)
    bakta_hits = spark.sql(f"""
        SELECT ba.gene_cluster_id, ba.gene, ba.product, ba.kegg_orthology_id,
               gc.gtdb_species_clade_id, gc.is_core, gc.is_auxiliary, gc.is_singleton
        FROM kbase_ke_pangenome.bakta_annotations ba
        JOIN kbase_ke_pangenome.gene_cluster gc ON ba.gene_cluster_id = gc.gene_cluster_id
        WHERE ba.gene IN ('{gene_in}')
        AND ba.gene IS NOT NULL
    """).toPandas()
    bakta_hits.to_csv(bakta_markers_path, index=False)
    print(f'Bakta marker hits: {len(bakta_hits):,}')

print(f'\nGene hits breakdown:')
print(bakta_hits['gene'].value_counts().head(20).to_string())

## 3. Query Pfam domains for secretion system markers

In [ ]:
# Pfam-based search for secretion systems and other markers
MARKER_PFAMS = {
    # T3SS needle complex
    'PF00771': 't3ss_needle',      # T3SS inner rod
    'PF01313': 't3ss_prgH',        # T3SS PrgH
    'PF01514': 't3ss_secretin',    # Secretin superfamily (also T2SS)
    # T4SS components
    'PF03135': 'virb8_t4ss',       # VirB8
    # T6SS components
    'PF05936': 't6ss_hcp',         # Hcp (T6SS tube)
    'PF05943': 't6ss_vgrg',        # VgrG (T6SS spike)
    # Pectate lyase (CWDE)
    'PF00544': 'pectate_lyase',    # Pectate lyase superfamily
    'PF12708': 'pectate_lyase_3',  # Pectate lyase 3
    # Cellulase (CWDE)
    'PF00150': 'cellulase',        # Cellulase (glycosyl hydrolase family 5)
    # Nitrogenase iron protein
    'PF00142': 'nitrogenase',      # Nitrogenase iron protein (NifH)
}

pfam_markers_path = os.path.join(DATA, 'pfam_marker_hits.csv')
if os.path.exists(pfam_markers_path):
    pfam_hits = pd.read_csv(pfam_markers_path)
    print(f'Pfam marker hits (loaded from cache): {len(pfam_hits):,}')
else:
    spark = get_spark_session()
    pfam_ids = "', '".join(MARKER_PFAMS.keys())
    pfam_hits = spark.sql(f"""
        SELECT bpf.gene_cluster_id, bpf.pfam_id, bpf.pfam_name, bpf.score, bpf.evalue,
               gc.gtdb_species_clade_id, gc.is_core, gc.is_auxiliary, gc.is_singleton
        FROM kbase_ke_pangenome.bakta_pfam_domains bpf
        JOIN kbase_ke_pangenome.gene_cluster gc ON bpf.gene_cluster_id = gc.gene_cluster_id
        WHERE bpf.pfam_id IN ('{pfam_ids}')
    """).toPandas()
    pfam_hits.to_csv(pfam_markers_path, index=False)
    print(f'Pfam marker hits: {len(pfam_hits):,}')

print(f'\nPfam domain hits:')
for pfam_id, label in MARKER_PFAMS.items():
    n = (pfam_hits['pfam_id'] == pfam_id).sum()
    print(f'  {pfam_id} ({label}): {n:,}')

## 4. Query KEGG KOs for secretion system modules

In [ ]:
# KEGG module-level search: T3SS (M00332), T4SS (M00333)
MARKER_KOS = {
    # T3SS core components
    'K03219': 't3ss_yscC', 'K03222': 't3ss_yscJ', 'K03224': 't3ss_yscN',
    'K03229': 't3ss_yscV', 'K03226': 't3ss_yscQ', 'K03227': 't3ss_yscR',
    # Nitrogenase
    'K02588': 'nifH_ko', 'K02586': 'nifD_ko', 'K02591': 'nifK_ko',
    # ACC deaminase
    'K01505': 'acdS_ko',
    # Siderophore enterobactin
    'K02362': 'entA_ko', 'K02363': 'entB_ko', 'K02364': 'entC_ko',
}

kegg_markers_path = os.path.join(DATA, 'kegg_marker_hits.csv')
if os.path.exists(kegg_markers_path):
    kegg_hits = pd.read_csv(kegg_markers_path)
    print(f'KEGG marker hits (loaded from cache): {len(kegg_hits):,}')
else:
    spark = get_spark_session()
    ko_likes = ' OR '.join([f"ann.KEGG_ko LIKE '%{ko}%'" for ko in MARKER_KOS.keys()])
    kegg_hits = spark.sql(f"""
        SELECT ann.query_name AS gene_cluster_id, ann.KEGG_ko, ann.Description,
               gc.gtdb_species_clade_id, gc.is_core, gc.is_auxiliary, gc.is_singleton
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.gene_cluster gc ON ann.query_name = gc.gene_cluster_id
        WHERE {ko_likes}
    """).toPandas()
    kegg_hits.to_csv(kegg_markers_path, index=False)
    print(f'KEGG marker hits: {len(kegg_hits):,}')

# Assign primary KO
def assign_kegg_marker(kegg_ko):
    if pd.isna(kegg_ko):
        return 'unknown'
    ko = str(kegg_ko)
    for ko_id, label in MARKER_KOS.items():
        if ko_id in ko:
            return label
    return 'unknown'

kegg_hits['marker_label'] = kegg_hits['KEGG_ko'].apply(assign_kegg_marker)
print(f'\nKEGG marker distribution:')
print(kegg_hits['marker_label'].value_counts().to_string())

## 5. Product description keyword search for additional pathogenicity markers

In [ ]:
# Search bakta_annotations.product for pathogenicity-related keywords
product_markers_path = os.path.join(DATA, 'product_marker_hits.csv')
if os.path.exists(product_markers_path):
    product_hits = pd.read_csv(product_markers_path)
    print(f'Product keyword hits (loaded from cache): {len(product_hits):,}')
else:
    spark = get_spark_session()
    product_hits = spark.sql("""
        SELECT ba.gene_cluster_id, ba.gene, ba.product,
               gc.gtdb_species_clade_id, gc.is_core, gc.is_auxiliary, gc.is_singleton
        FROM kbase_ke_pangenome.bakta_annotations ba
        JOIN kbase_ke_pangenome.gene_cluster gc ON ba.gene_cluster_id = gc.gene_cluster_id
        WHERE (
            LOWER(ba.product) LIKE '%type iii secretion%'
            OR LOWER(ba.product) LIKE '%effector protein%'
            OR LOWER(ba.product) LIKE '%coronatine%'
            OR LOWER(ba.product) LIKE '%pectate lyase%'
            OR LOWER(ba.product) LIKE '%pectinase%'
            OR LOWER(ba.product) LIKE '%cellulase%'
            OR LOWER(ba.product) LIKE '%xylanase%'
            OR LOWER(ba.product) LIKE '%phytotoxin%'
            OR LOWER(ba.product) LIKE '%plant pathogen%'
            OR LOWER(ba.product) LIKE '%hrp pilus%'
            OR LOWER(ba.product) LIKE '%avirulence%'
            OR LOWER(ba.product) LIKE '%type vi secretion%'
        )
    """).toPandas()
    product_hits.to_csv(product_markers_path, index=False)
    print(f'Product keyword hits: {len(product_hits):,}')

# Classify product hits by category
def classify_product(product):
    if pd.isna(product):
        return 'unknown'
    p = str(product).lower()
    if 'type iii secretion' in p or 'hrp' in p:
        return 't3ss_product'
    if 'type vi secretion' in p:
        return 't6ss_product'
    if 'effector' in p or 'avirulence' in p:
        return 'effector'
    if 'pectate lyase' in p or 'pectinase' in p:
        return 'cwde_pectinase'
    if 'cellulase' in p or 'xylanase' in p:
        return 'cwde_cellulase'
    if 'coronatine' in p or 'phytotoxin' in p:
        return 'phytotoxin'
    return 'other_pathogenic'

product_hits['marker_category'] = product_hits['product'].apply(classify_product)
print(f'\nProduct-based marker categories:')
print(product_hits['marker_category'].value_counts().to_string())

## 6. Consolidate all marker hits

In [ ]:
# Map bakta gene hits to functional categories and cohort types
def get_functional_category(gene):
    if gene in PGP_GENES:
        return PGP_GENES[gene]
    if gene in PATHOGEN_GENES:
        return PATHOGEN_GENES[gene]
    if gene in COLONIZATION_GENES:
        return COLONIZATION_GENES[gene]
    return 'unknown'

def get_cohort_type(gene):
    if gene in PGP_GENES:
        return 'beneficial'
    if gene in PATHOGEN_GENES:
        return 'pathogenic'
    if gene in COLONIZATION_GENES:
        return 'colonization'
    return 'unknown'

bakta_hits['functional_category'] = bakta_hits['gene'].apply(get_functional_category)
bakta_hits['cohort_type'] = bakta_hits['gene'].apply(get_cohort_type)
bakta_hits['source'] = 'bakta_gene'

# Standardize columns for concatenation
cols = ['gene_cluster_id', 'gtdb_species_clade_id', 'is_core', 'is_auxiliary', 'is_singleton',
        'functional_category', 'cohort_type', 'source']

# Pfam hits
pfam_hits['functional_category'] = pfam_hits['pfam_id'].map(MARKER_PFAMS)
pfam_hits['cohort_type'] = pfam_hits['functional_category'].apply(
    lambda x: 'pathogenic' if 't3ss' in str(x) or 't4ss' in str(x) or 't6ss' in str(x)
    or 'lyase' in str(x) or 'cellulase' in str(x)
    else 'beneficial' if 'nitrogenase' in str(x)
    else 'unknown'
)
pfam_hits['source'] = 'pfam_domain'

# Product hits
product_hits['functional_category'] = product_hits['marker_category']
product_hits['cohort_type'] = 'pathogenic'
product_hits['source'] = 'product_keyword'

# Concatenate all sources (deduplicate by gene_cluster_id)
all_markers = pd.concat([
    bakta_hits[cols],
    pfam_hits[cols],
    product_hits[cols],
], ignore_index=True)

# Deduplicate: keep one entry per gene_cluster_id with priority bakta_gene > pfam > product
source_priority = {'bakta_gene': 0, 'pfam_domain': 1, 'product_keyword': 2}
all_markers['source_priority'] = all_markers['source'].map(source_priority)
all_markers = all_markers.sort_values('source_priority').drop_duplicates('gene_cluster_id', keep='first')
all_markers = all_markers.drop('source_priority', axis=1)

print(f'Total unique marker gene clusters: {len(all_markers):,}')
print(f'Unique species with markers: {all_markers["gtdb_species_clade_id"].nunique():,}')
print(f'\nBy cohort type:')
print(all_markers['cohort_type'].value_counts().to_string())
print(f'\nBy functional category (top 20):')
print(all_markers['functional_category'].value_counts().head(20).to_string())

## 7. Build species-level marker matrix

In [ ]:
# Build species × functional_category presence/absence matrix
species_markers = all_markers.groupby(['gtdb_species_clade_id', 'functional_category']).agg(
    n_clusters=('gene_cluster_id', 'nunique'),
    n_core=('is_core', 'sum'),
    n_singleton=('is_singleton', 'sum'),
).reset_index()

# Pivot to wide format (presence/absence)
marker_presence = species_markers.pivot_table(
    index='gtdb_species_clade_id', columns='functional_category',
    values='n_clusters', fill_value=0
).reset_index()

# Create boolean columns
func_categories = [c for c in marker_presence.columns if c != 'gtdb_species_clade_id']
for cat in func_categories:
    marker_presence[f'{cat}_present'] = (marker_presence[cat] > 0).astype(int)

# Count PGP and pathogenic markers per species
pgp_categories = set(PGP_GENES.values())
pathogen_categories = set(PATHOGEN_GENES.values())

pgp_cols = [f'{c}_present' for c in pgp_categories if f'{c}_present' in marker_presence.columns]
path_cols = [f'{c}_present' for c in pathogen_categories if f'{c}_present' in marker_presence.columns]

marker_presence['n_pgp_functions'] = marker_presence[pgp_cols].sum(axis=1) if pgp_cols else 0
marker_presence['n_pathogen_functions'] = marker_presence[path_cols].sum(axis=1) if path_cols else 0

print(f'Species marker matrix: {len(marker_presence):,} species × {len(func_categories)} functions')
print(f'\nPGP function richness: mean={marker_presence["n_pgp_functions"].mean():.1f}, '
      f'max={marker_presence["n_pgp_functions"].max()}')
print(f'Pathogen function richness: mean={marker_presence["n_pathogen_functions"].mean():.1f}, '
      f'max={marker_presence["n_pathogen_functions"].max()}')

## 8. Classify species into PGP/pathogen/dual-nature cohorts

In [ ]:
# Classify species into cohorts based on marker gene presence
def assign_marker_cohort(row):
    has_pgp = row['n_pgp_functions'] > 0
    has_path = row['n_pathogen_functions'] > 0
    if has_pgp and has_path:
        return 'dual_nature'
    elif has_pgp:
        return 'pgp_only'
    elif has_path:
        return 'pathogen_only'
    else:
        return 'neutral'

marker_presence['marker_cohort'] = marker_presence.apply(assign_marker_cohort, axis=1)

print('Species cohort distribution (marker-based):')
print(marker_presence['marker_cohort'].value_counts().to_string())

# Report dual-nature species
dual = marker_presence[marker_presence['marker_cohort'] == 'dual_nature']
print(f'\nDual-nature species: {len(dual):,}')
print(f'  Mean PGP functions: {dual["n_pgp_functions"].mean():.1f}')
print(f'  Mean pathogen functions: {dual["n_pathogen_functions"].mean():.1f}')

## 9. Validation: spot-check known PGP and pathogenic organisms

In [ ]:
# Load species compartment data from NB01
species_comp = pd.read_csv(os.path.join(DATA, 'species_compartment.csv'))

# Merge marker data with taxonomy
marker_with_tax = marker_presence.merge(
    species_comp[['gtdb_species_clade_id', 'dominant_compartment', 'genus', 'phylum']],
    on='gtdb_species_clade_id', how='left'
)

# Spot-check known PGPB
known_pgpb = ['Rhizobium', 'Azospirillum', 'Bacillus', 'Pseudomonas', 'Mesorhizobium']
print('=== PGPB Validation ===')
for genus in known_pgpb:
    sub = marker_with_tax[marker_with_tax['genus'].str.contains(genus, na=False)]
    if len(sub) > 0:
        n_pgp = (sub['n_pgp_functions'] > 0).sum()
        print(f'  {genus:20s}: {len(sub):4d} species, {n_pgp} with PGP markers, '
              f'median PGP functions={sub["n_pgp_functions"].median():.0f}')

# Spot-check known plant pathogens
known_pathogens = ['Ralstonia', 'Xanthomonas', 'Erwinia', 'Pectobacterium', 'Agrobacterium']
print('\n=== Plant Pathogen Validation ===')
for genus in known_pathogens:
    sub = marker_with_tax[marker_with_tax['genus'].str.contains(genus, na=False)]
    if len(sub) > 0:
        n_path = (sub['n_pathogen_functions'] > 0).sum()
        cohorts = sub['marker_cohort'].value_counts().to_dict()
        print(f'  {genus:20s}: {len(sub):4d} species, {n_path} with pathogenic markers, '
              f'cohorts={cohorts}')

## 10. Summary visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: marker gene cluster counts by functional category
ax = axes[0]
cat_counts = all_markers['functional_category'].value_counts().head(15).sort_values()
cat_colors = []
for cat in cat_counts.index:
    if cat in pgp_categories:
        cat_colors.append('#4CAF50')
    elif cat in pathogen_categories:
        cat_colors.append('#F44336')
    else:
        cat_colors.append('#2196F3')
ax.barh(cat_counts.index, cat_counts.values, color=cat_colors)
ax.set_xlabel('Number of gene clusters')
ax.set_title('Marker gene clusters by function')

# Middle: species cohort pie chart
ax = axes[1]
cohort_counts = marker_presence['marker_cohort'].value_counts()
cohort_colors = {'pgp_only': '#4CAF50', 'pathogen_only': '#F44336',
                 'dual_nature': '#FF9800', 'neutral': '#9E9E9E'}
ax.pie(cohort_counts.values,
       labels=[f'{k}\n({v:,})' for k, v in cohort_counts.items()],
       colors=[cohort_colors.get(k, '#9E9E9E') for k in cohort_counts.index],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Species by marker cohort')

# Right: core/accessory/singleton fraction by cohort type
ax = axes[2]
arch_by_cohort = all_markers.groupby('cohort_type').agg(
    pct_core=('is_core', 'mean'),
    pct_singleton=('is_singleton', 'mean'),
).reset_index()
arch_by_cohort['pct_accessory'] = 1 - arch_by_cohort['pct_core'] - arch_by_cohort['pct_singleton']

x = np.arange(len(arch_by_cohort))
ax.bar(x, arch_by_cohort['pct_core'] * 100, label='Core', color='#2196F3')
ax.bar(x, arch_by_cohort['pct_accessory'] * 100, bottom=arch_by_cohort['pct_core'] * 100,
       label='Accessory', color='#FF9800')
ax.bar(x, arch_by_cohort['pct_singleton'] * 100,
       bottom=(arch_by_cohort['pct_core'] + arch_by_cohort['pct_accessory']) * 100,
       label='Singleton', color='#9E9E9E')
ax.set_xticks(x)
ax.set_xticklabels(arch_by_cohort['cohort_type'], fontsize=9)
ax.set_ylabel('Percentage')
ax.set_title('Genomic architecture by marker type')
ax.legend(fontsize=8)
ax.axhline(46.8, color='black', linestyle='--', alpha=0.5, label='Genome-wide core baseline')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'nb02_marker_survey.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/nb02_marker_survey.png')

## 11. Save outputs

In [ ]:
all_markers.to_csv(os.path.join(DATA, 'marker_gene_clusters.csv'), index=False)
marker_presence.to_csv(os.path.join(DATA, 'species_marker_matrix.csv'), index=False)

# Save cohort assignment with taxonomy
cohort_summary = marker_with_tax[['gtdb_species_clade_id', 'marker_cohort',
                                   'n_pgp_functions', 'n_pathogen_functions',
                                   'genus', 'phylum', 'dominant_compartment']].copy()
cohort_summary.to_csv(os.path.join(DATA, 'species_cohort_markers.csv'), index=False)

print('=== NB02 Summary ===')
print(f'Total marker gene clusters: {len(all_markers):,}')
print(f'Species with markers: {all_markers["gtdb_species_clade_id"].nunique():,}')
print(f'Cohort distribution: {marker_presence["marker_cohort"].value_counts().to_dict()}')
print(f'\nOutputs saved to {DATA}/')
print('  - marker_gene_clusters.csv')
print('  - species_marker_matrix.csv')
print('  - species_cohort_markers.csv')
print(f'\nReady for NB03 (enrichment analysis) and NB04 (compartment profiling)')